# qvarnet tutorial — one training run, every API call explained

This notebook runs **a single VMC training** of the Calogero-Sutherland model and explains
*every* qvarnet API call: what each object is, what each argument means, and — at the end —
**how to pull every result out of the returned object**.

We deliberately keep the physics minimal (small `N`, the exact single-parameter ansatz) so the
focus stays on the API, not on convergence tricks. For the richer CS study see
`../calogero-sutherland/calogero_sutherland.ipynb`.

### The pipeline in one sentence
`train(...)` takes a **model** (the wavefunction ansatz), a **hamiltonian** (defines the energy),
an **optimizer**, a **sampler** config and a **training** config, runs Metropolis-Hastings VMC for
`n_epochs`, and returns a `TrainResult` holding the per-epoch **history** and the best/final
**parameters**.

## 1. Imports — what each one is for

In [ ]:
%matplotlib inline
import numpy as np
import jax
import jax.numpy as jnp
import optax                        # optimizers (Adam, SGD, schedules)
import matplotlib.pyplot as plt

# --- the five things train() needs ---
from qvarnet.train import train                       # the entry point
from qvarnet.config.training_setup import TrainingConfig   # run-level config (epochs, seed, ...)
from qvarnet.config.coord_mode import LabCoords        # coordinate convention (lab vs Jacobi)
from qvarnet.hamiltonian.continuous import CalogeroSutherlandHamiltonian  # the energy operator

# --- the model: a neural-network wavefunction composed with LogWavefunction ---
from qvarnet.models.compose import LogWavefunction     # log|psi| = network + envelope + jastrow
from qvarnet.models.mlp import MLP                     # the trainable neural network
from qvarnet.models.envelopes import GaussianEnvelope  # confining factor
from qvarnet.models.jastrow import LogJastrow          # two-body correlation / cusp
from qvarnet.boundaries import NoBoundary              # open-boundary coordinate transform
from qvarnet.models.analytic import CalogeroSutherlandAnalyticModel  # exact baseline (optional)

# --- only needed for the 'evaluate an observable' section at the very end ---
from qvarnet.vmc.probability import build_prob_fn      # wraps log|psi| -> a |psi|^2 prob fn
from qvarnet.samplers import sample_and_process        # draw MCMC samples from a trained psi

## 2. The physical system

CS model, `N` particles on a line in a harmonic trap, convention $\hbar^2/m=1$:

$$H = -\sum_i\partial_i^2 + \sum_i x_i^2 + 2L(L-1)\sum_{i<j}\frac{1}{(x_i-x_j)^2}$$

We pick numbers and compute the **exact** ground-state energy $E_0=N(1+L(N-1))$ so we have a
target to compare the VMC result against.

In [ ]:
N      = 4      # particles (small, so the run is fast)
N_DIM  = 1      # CS lives in 1D
L      = 1.5    # CS coupling. The exact wavefunction has Jastrow exponent lambda = L.
DoF    = N * N_DIM

E_exact = N * (1 + L * (N - 1))   # exact total energy in this convention
print(f'N={N}, L={L}  ->  exact E0 = {E_exact:.4f}  (E0/N = {E_exact/N:.4f})')

## 3. The Hamiltonian

`CalogeroSutherlandHamiltonian` is the energy operator. `train()` uses it only through its
`local_energy(params, samples, apply)` method — you never call that yourself.

Arguments:
- `L` — the CS coupling; sets the interaction strength $g=2L(L-1)$.
- `epsilon` — softens the $1/(x_i-x_j)^2$ singularity (`1/((x_i-x_j)^2+epsilon)`). Small but
  nonzero keeps the local energy finite when two walkers nearly coincide.

(It also carries `omega_trap`, `pairwise_impl`, and a `laplacian_method` — all sensible defaults.)

In [ ]:
hamiltonian = CalogeroSutherlandHamiltonian(L=L, epsilon=1e-4)
print(hamiltonian)

## 4. The model — a neural-network wavefunction via `LogWavefunction`

This is the heart of an NQS package. A *model* maps a configuration `x` of shape `(..., N)` to a
single scalar **`log|psi(x)|`** of shape `(..., 1)` — qvarnet works in log-space for stability.

`LogWavefunction` is the composable ansatz. It **sums three log-space pieces**:

$$\log|\psi(x)| \;=\; \underbrace{\mathrm{network}(\mathrm{transform}(x))}_{(...,\,1)}
\;+\; \underbrace{\mathrm{envelope}(x)}_{(...,\,1)}
\;+\; \underbrace{\mathrm{jastrow}(x)}_{(...,\,1)}$$

Each slot is independently swappable:

- **`transform`** — coordinate encoding / boundary. `NoBoundary()` passes raw coordinates (open
  system); `PeriodicBoundary(L)` gives a periodic sin/cos encoding for a box.
- **`network`** — the **trainable neural network**, the expressive core. Here `MLP(hidden=[64])`.
  Could instead be `DeepSet` (permutation-invariant), `TransformerWavefunction`, or a fermionic
  Slater net. (For per-particle networks like `DeepSet`, pass `n_particles=` to `LogWavefunction`
  so it reshapes the input to `(..., N, ppd)`.)
- **`envelope`** — a confining factor, `GaussianEnvelope()` $=-\alpha^2\sum_i x_i^2$ (learnable
  $\alpha$). It gives `psi` the right decay so it is normalisable in the trap.
- **`jastrow`** — an explicit two-body factor, `LogJastrow` $=\lambda\sum_{i<j}\log|x_i-x_j|$. It
  builds in the interaction **cusp** that a smooth MLP cannot represent — exactly the CS correlation.

The MLP weights, the envelope's $\alpha$, and the jastrow's $\lambda$ are **all learned together**.
The exact CS ground state already lives in the envelope+jastrow part (at $\alpha^2=\tfrac12$,
$\lambda=L$); the network adds flexible corrections on top. A short run of a neural ansatz lands
*close* to `E_exact` — and being variational, the energy is always an **upper bound** (it will not
be exact, unlike the single-parameter `CalogeroSutherlandAnalyticModel` baseline).

In [ ]:
from flax.traverse_util import flatten_dict   # for readable pytree printing

# The headline ansatz: a neural network composed with an envelope and a Jastrow factor.
#   log|psi(x)| = network(transform(x)) + envelope(x) + jastrow(x)     (each shape (..., 1))
model = LogWavefunction(
    transform=NoBoundary(),                               # open boundary; raw coordinates
    network=MLP(hidden=[64]),                             # the trainable neural network
    envelope=GaussianEnvelope(),                          # -alpha^2 * sum_i x_i^2   (learnable alpha)
    jastrow=LogJastrow(n_particles=N, lambda_init=1.0),   # lambda * sum_{i<j} log|x_i - x_j|
)

# Exact single-parameter baseline (uncomment to compare):
# model = CalogeroSutherlandAnalyticModel(lambda_init=0.5)

# Initialise and inspect: the parameter pytree holds the MLP weights AND the envelope alpha
# AND the jastrow lambda — every one of them is trained together by train().
dummy = jnp.ones((3, DoF))                               # 3 configurations of N particles
init_params = model.init(jax.random.PRNGKey(0), dummy)
print('parameter pytree (name -> shape):')
for key, leaf in flatten_dict(init_params).items():
    print(f"   {'/'.join(key):28s} {np.asarray(leaf).shape}")
n_scalars = sum(np.asarray(l).size for l in jax.tree_util.tree_leaves(init_params))
print(f'total trainable scalars: {n_scalars}')
print('log|psi| shape:', model.apply(init_params, dummy).shape, '(one scalar per configuration)')

## 5. The optimizer

Any [optax](https://optax.readthedocs.io) optimizer works. `train()` applies it to the gradient of
the VMC energy w.r.t. the model parameters. Plain Adam at a modest learning rate is fine here.

(If you instead set `TrainingConfig(use_qgt=True)`, the optimizer you pass is ignored and Stochastic
Reconfiguration / natural-gradient SGD is used — not needed for this tutorial.)

In [ ]:
optimizer = optax.adam(learning_rate=2e-3)

## 6. The sampler configuration

VMC estimates $\langle E\rangle$ by averaging the *local energy* over configurations drawn from
$|\psi|^2$ with Metropolis-Hastings. You pass the sampler settings as a **plain dict**
(`train()` parses it into a `SamplingConfig`). The keys:

- `step_size` — MH proposal width. Adapted automatically when `is_update_step_size=True` (below).
- `chain_length` — total MH steps taken per epoch, **per chain**.
- `thermalization_steps` — burn-in steps discarded before collecting samples (must be `< chain_length`).
- `thinning_factor` — keep every k-th post-burn-in step to reduce autocorrelation.
- *(omitted here)* `box_L` — set this to a box side to enable the periodic sampler. Absent = open
  boundary, which is what we want (the harmonic trap confines the walkers).

The number of **chains** (parallel walkers) is not set here — it is the first entry of `shape` in
the `train()` call below.

In [ ]:
sampler_params = {
    'step_size':            0.4,
    'chain_length':         21,
    'thermalization_steps': 20,
    'thinning_factor':      1,
    # no 'box_L' -> open boundary
}

## 7. The training configuration

`TrainingConfig` holds run-level (not per-step) settings. The ones we use:

- `n_epochs` — number of optimization steps.
- `rng_seed` — seeds parameter init **and** the MCMC, so the whole run is reproducible.
- `warm_walkers=True` — carry the final walker positions of one epoch into the next (don't reset
  the chains every epoch). Cheaper and better-mixed.
- `is_update_step_size=True` — adapt `step_size` toward `target_acceptance` (default 0.5),
  clamped to `[min_step, max_step]`.
- `checkpoint_path` — where checkpoints would be written. (Writing only happens if
  `save_checkpoints=True`; resuming happens automatically if a checkpoint already exists there.)

In [ ]:
training_config = TrainingConfig(
    n_epochs=1000,            # a neural ansatz needs more steps than the 1-parameter baseline
    rng_seed=0,
    warm_walkers=True,
    is_update_step_size=True,
    min_step=1e-4,
    max_step=5.0,
    checkpoint_path='./checkpoints/tutorial',
)

## 8. The one `train()` call — every argument

This is the single training run. Arguments:

- `shape=(n_chains, DoF)` — **`n_chains`** parallel MCMC walkers, each a vector of length
  `DoF = N*N_DIM`. This is where the number of chains (batch size) is set.
- `model` — the ansatz from step 4.
- `optimizer` — from step 5.
- `hamiltonian` — from step 3 (defines the energy being minimised).
- `training_config` — from step 7.
- `sampler_params` — from step 6.
- `coord_mode=LabCoords()` — interpret samples as lab coordinates (the alternative, `JacobiCoords`,
  removes the centre of mass; not needed here).
- `select='energy'` and `k_best=3` — retain the parameter sets from the 3 epochs with the lowest
  **energy**, exposed afterwards as `best_params()` / `best_k_params()`. (Default `select` is `'std'`.)

It returns a single `TrainResult`.

In [ ]:
N_CHAINS = 2048

result = train(
    shape=(N_CHAINS, DoF),
    model=model,
    optimizer=optimizer,
    hamiltonian=hamiltonian,
    training_config=training_config,
    sampler_params=sampler_params,
    coord_mode=LabCoords(),
    select='energy',   # rank retained snapshots by lowest energy
    k_best=3,          # keep the 3 best parameter sets
)
result   # __repr__ shows n_steps and the last energy

# 9. Extracting the results

Everything the run produced lives on the returned `result` (`TrainResult`). The three things you
will want: the **history** (per-epoch metrics), the **parameters** (best / final), and a
**convergence verdict**. We go through each.

### 9a. The history — per-epoch metrics

`result.history` is a `MetricsHistory`. Two equivalent ways to read it:

1. **Iterate** it: each item is an `EpochRecord` with attribute access (`s.energy`, `s.std`, ...).
2. **Stack a field**: `result.history.get('energy')` returns a `(n_epochs,)` numpy array.

Available per-epoch fields: `step`, `energy` (⟨E⟩), `std` (σ of the local energy over the batch),
`error_of_mean` (σ/√M), `acceptance_rate` (per-chain, a vector), `step_size`, `cm_mean`/`cm_std`
(centre-of-mass diagnostics), `wall_time`. **No parameters are stored in the history** (kept light).

In [ ]:
hist = result.history
print('n_epochs        :', len(hist))
print('fields per epoch:', list(hist[-1].keys()))
print()

# Way 1 — iterate / index. Last epoch:
last = hist[-1]
print(f'last epoch: step={last.step}  E={float(last.energy):.4f}  '
      f'std={float(last.std):.4f}  step_size={float(last.step_size):.3f}')

# Way 2 — stacked arrays for analysis/plots:
E   = hist.get('energy')          # (n_epochs,)
sig = hist.get('std')             # (n_epochs,)
acc = hist.get('acceptance_rate') # (n_epochs, n_chains) -> average over chains
print('energy array shape:', E.shape, '| acceptance array shape:', acc.shape)

### 9b. A robust final-energy estimate

The instantaneous last-epoch energy is noisy. The usual estimate is the **mean over a tail** of
epochs (after convergence), with the error as the tail's standard error.

In [ ]:
tail = E[-150:]
E_est  = float(tail.mean())
E_err  = float(tail.std() / np.sqrt(len(tail)))
print(f'VMC energy = {E_est:.4f} +/- {E_err:.4f}')
print(f'exact      = {E_exact:.4f}')
print(f'gap        = {E_est - E_exact:+.4f}')

### 9c. Plot the convergence

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(E, lw=0.7)
ax[0].axhline(E_exact, color='k', ls='--', lw=0.9, label='exact $E_0$')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('energy'); ax[0].legend(); ax[0].set_title('energy')
ax[1].plot(acc.mean(axis=1), lw=0.7, color='C1')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('mean acceptance'); ax[1].set_title('MH acceptance')
plt.tight_layout(); plt.savefig('tutorial_convergence.png', dpi=120); plt.show()

### 9d. The trained parameters

Parameters are **not** in the history; they are exposed separately:

- `result.final_params` — params at the last epoch (always present).
- `result.best_params()` — params of the single best retained snapshot (by the `select` metric),
  falling back to `final_params` if `k_best=0`.
- `result.best_k_params(n)` / `result.best_k(n)` / `result.best_steps(n)` — the top-`n` snapshots,
  their `{step, metric, params}` dicts, and the epochs they came from.

The pytree now holds the **whole network** plus the two interpretable physical parameters
`envelope/alpha` and `jastrow/lambda`. We count the parameters and pull those two out (by name,
wherever they sit in the tree) to check they moved toward the exact values
($\alpha^2\to\tfrac12$, $\lambda\to L$).

In [ ]:
best = result.best_params()
flat = flatten_dict(best)                       # {('params','jastrow','lambda'): value, ...}

print('total trained scalars:', sum(np.asarray(v).size for v in flat.values()),
      '(MLP weights + envelope alpha + jastrow lambda)')

print('\ninterpretable physical parameters:')
for key, val in flat.items():
    if key[-1] in ('lambda', 'alpha'):         # unique to the jastrow / envelope
        print(f"   {'/'.join(key):28s} = {float(np.asarray(val)):.4f}")
print(f"   targets: alpha -> {0.5**0.5:.3f}  (alpha^2 = 1/2),   lambda -> L = {L}")

# Generic access that works for ANY model: just the flat list of leaf arrays.
print('\nnumber of parameter tensors (leaves):', len(jax.tree_util.tree_leaves(best)))

# The top-k snapshots and which epoch each came from:
print('best 3 epochs (lowest energy):', result.best_steps(3))
for snap in result.best_k(3):
    print(f"  step={snap['step']:4d}  metric(energy)={snap['metric']:.4f}")

### 9e. Rank epochs and get a convergence verdict

- `result.best(n, metric)` ranks the **history records** (lightweight, no params) by a metric
  (`'energy'`, `'std'`, or a callable). Useful to find the lowest-energy epochs.
- `result.diagnose()` runs qvarnet's three-referee check (stationary? at the Monte-Carlo floor?
  chains mixed?) and prints a formatted report — the standard end-of-run sanity check.

In [ ]:
top = result.best(n=3, metric='energy')   # 3 lowest-energy epochs
print('lowest-energy epochs:')
for s in top:
    print(f'  step={s.step:4d}  E={float(s.energy):.4f}  std={float(s.std):.4f}')
print()
verdict = result.diagnose()   # prints the report; also returns a dict

### 9f. (Bonus) Use the trained wavefunction to measure an observable

`best_params()` is a frozen wavefunction you can sample from to compute *any* observable. We draw
fresh configurations from $|\psi|^2$ and estimate the per-particle $\langle x^2\rangle$ and the
one-body density.

- `build_prob_fn(model.apply)` turns `log|psi|` into the probability function the sampler needs.
- `sample_and_process(...)` runs MH and returns samples of shape `(n_kept, DoF)`.

In [ ]:
prob_fn = build_prob_fn(model.apply)
x0 = jax.random.normal(jax.random.PRNGKey(7), (1024, DoF))   # start near the trap centre
samples, _, _ = sample_and_process(
    key=jax.random.PRNGKey(8), prob_fn=prob_fn, prob_params=best,
    init_positions=x0, step_size=0.4, n_chains=1024, dof=DoF,
    n_steps=400, burn_in=150, thinning=2, box_L=0.0,   # box_L=0 -> open boundary
)
pos = np.asarray(samples).reshape(-1, N)        # (n_configs, N)
print('drew', pos.shape[0], 'configurations')
print(f'<x^2> per particle = {np.mean(pos**2):.4f}')

plt.figure(figsize=(6, 3))
plt.hist(pos.ravel(), bins=60, density=True, alpha=0.8)
plt.xlabel('x'); plt.ylabel('one-body density n(x)')
plt.title('density of the trained CS ground state')
plt.tight_layout(); plt.savefig('tutorial_density.png', dpi=120); plt.show()

## Recap — the API surface you just used

| Call | Role |
|---|---|
| `CalogeroSutherlandHamiltonian(L, epsilon)` | defines the energy operator |
| `LogWavefunction(transform, network, envelope, jastrow)` | composes the ansatz `log|psi|` |
| `MLP(hidden=[...])` / `GaussianEnvelope()` / `LogJastrow(...)` | the network / envelope / Jastrow pieces |
| `NoBoundary()` / `PeriodicBoundary(L)` | the coordinate transform (open / periodic) |
| `model.init(key, x)` / `model.apply(params, x)` | initialise / evaluate the ansatz |
| `optax.adam(lr)` | the optimizer |
| `sampler_params` dict | MH sampler settings (`step_size`, `chain_length`, ...) |
| `TrainingConfig(...)` | run-level settings (`n_epochs`, seed, adaptation, ...) |
| `train(shape, model, optimizer, hamiltonian, ...)` | runs the VMC optimization |
| `result.history` (`.get(field)`, iterate) | per-epoch metrics |
| `result.final_params` / `best_params()` / `best_k_params(n)` | trained parameters (full pytree) |
| `result.best(n, metric)` / `result.diagnose()` | rank epochs / convergence verdict |
| `build_prob_fn` + `sample_and_process` | sample the trained `psi` for observables |

> Swap `network=MLP(...)` for `DeepSet`, `TransformerWavefunction`, or a fermionic net — everything
> else (training, extraction, observables) stays identical. See `qvarnet_tour.ipynb` for the full menu.